##Snowflake - Data Ingestion

This notebook parses and ingests the data into Snowflake. Unfortunately, there will be some manual steps to do in order to make this work. 

This notebook assumes the following manual steps have been executed:
- copy the generated data from the Databricks volume, to an S3 location
- create a staging in Snowflake 

In [ ]:
%run "./helpers/Helper Functions"

In [ ]:
# --- Create Widgets (text) ---
dbutils.widgets.removeAll()
dbutils.widgets.text("sf_database", "", "Snowflake Database")
dbutils.widgets.text("sf_schema", "", "Snowflake Schema")
dbutils.widgets.text("sf_staging", "", "Snowflake Staging Name")
dbutils.widgets.text("scale", "100", "Scale Factor (GB)")
dbutils.widgets.text("table_prefix", "", "Table Prefix (e.g. M_100_)")
dbutils.widgets.dropdown("sf_strategy", "native_gen1",
    ["native_gen1","native_gen2","iceberg_gen1","iceberg_gen2"], "Snowflake Strategy")
dbutils.widgets.text("sf_strategy_slug", "", "Snowflake Strategy Slug (auto-computed if blank)")
dbutils.widgets.text("dbx_catalog", "tpcds_sf_catalog", "Databricks Catalog (for volume)")
dbutils.widgets.text("dbx_schema", "tpcds", "Databricks Schema (for volume)")
dbutils.widgets.text("dbx_volume", "tpcds_raw_data", "Volume name")

In [ ]:
import os
# --- Retrieve Values, Set Env Vars, and Build Dictionary ---

params = dbutils.widgets.getAll()

for k,v in params.items():
    os.environ[f"W_{k.upper()}"] = v
    
# --- Validation Check (all required) ---

if params["sf_database"] == "":
    raise RuntimeError("Validation Error: Parameter 'sf_database' is required but is empty.")

if params["sf_schema"] == "":
    raise RuntimeError("Validation Error: Parameter 'sf_schema' is required but is empty.")

if params["sf_staging"] == "":
    raise RuntimeError("Validation Error: Parameter 'sf_staging' is required but is empty.")


In [ ]:
database = params['sf_database']
schema = params['sf_schema']
stage = params['sf_staging']
scale = params['scale']

ddl_statements = {
    "create_call_center": f"CREATE OR REPLACE TABLE {database}.{schema}.call_center            ( cc_call_center_sk int , cc_call_center_id varchar(16) , cc_rec_start_date date , cc_rec_end_date date , cc_closed_date_sk int , cc_open_date_sk int , cc_name varchar(50) , cc_class varchar(50) , cc_employees int , cc_sq_ft int , cc_hours varchar(20) , cc_manager varchar(40) , cc_mkt_id int , cc_mkt_class varchar(50) , cc_mkt_desc varchar(100) , cc_market_manager varchar(40) , cc_division int , cc_division_name varchar(50) , cc_company int , cc_company_name varchar(50) , cc_street_number varchar(10) , cc_street_name varchar(60) , cc_street_type varchar(15) , cc_suite_number varchar(10) , cc_city varchar(60) , cc_county varchar(30) , cc_state varchar(2) , cc_zip varchar(10) , cc_country varchar(20) , cc_gmt_offset decimal(5,2) , cc_tax_percentage decimal(5,2) ) CLUSTER BY(cc_call_center_sk) ;",
    "create_catalog_page": f"CREATE OR REPLACE TABLE {database}.{schema}.catalog_page           ( cp_catalog_page_sk int , cp_catalog_page_id varchar(16) , cp_start_date_sk int , cp_end_date_sk int , cp_department varchar(50) , cp_catalog_number int , cp_catalog_page_number int , cp_description varchar(100) , cp_type varchar(100) ) CLUSTER BY(cp_catalog_page_sk) ;",
    "create_catalog_returns": f"CREATE OR REPLACE TABLE {database}.{schema}.catalog_returns        ( cr_returned_date_sk int , cr_returned_time_sk int , cr_item_sk int , cr_refunded_customer_sk int , cr_refunded_cdemo_sk int , cr_refunded_hdemo_sk int , cr_refunded_addr_sk int , cr_returning_customer_sk int , cr_returning_cdemo_sk int , cr_returning_hdemo_sk int , cr_returning_addr_sk int , cr_call_center_sk int , cr_catalog_page_sk int , cr_ship_mode_sk int , cr_warehouse_sk int , cr_reason_sk int , cr_order_number bigint , cr_return_quantity int , cr_return_amount decimal(7,2) , cr_return_tax decimal(7,2) , cr_return_amt_inc_tax decimal(7,2) , cr_fee decimal(7,2) , cr_return_ship_cost decimal(7,2) , cr_refunded_cash decimal(7,2) , cr_reversed_charge decimal(7,2) , cr_store_credit decimal(7,2) , cr_net_loss decimal(7,2) ) CLUSTER BY(cr_returned_date_sk) ;",
    "create_catalog_sales": f"CREATE OR REPLACE TABLE {database}.{schema}.catalog_sales          ( cs_sold_date_sk int , cs_sold_time_sk int , cs_ship_date_sk int , cs_bill_customer_sk int , cs_bill_cdemo_sk int , cs_bill_hdemo_sk int , cs_bill_addr_sk int , cs_ship_customer_sk int , cs_ship_cdemo_sk int , cs_ship_hdemo_sk int , cs_ship_addr_sk int , cs_call_center_sk int , cs_catalog_page_sk int , cs_ship_mode_sk int , cs_warehouse_sk int , cs_item_sk int , cs_promo_sk int , cs_order_number bigint , cs_quantity int , cs_wholesale_cost decimal(7,2) , cs_list_price decimal(7,2) , cs_sales_price decimal(7,2) , cs_ext_discount_amt decimal(7,2) , cs_ext_sales_price decimal(7,2) , cs_ext_wholesale_cost decimal(7,2) , cs_ext_list_price decimal(7,2) , cs_ext_tax decimal(7,2) , cs_coupon_amt decimal(7,2) , cs_ext_ship_cost decimal(7,2) , cs_net_paid decimal(7,2) , cs_net_paid_inc_tax decimal(7,2) , cs_net_paid_inc_ship decimal(7,2) , cs_net_paid_inc_ship_tax decimal(7,2) , cs_net_profit decimal(7,2) ) CLUSTER BY(cs_sold_date_sk) ;",
    "create_customer": f"CREATE OR REPLACE TABLE {database}.{schema}.customer               ( c_customer_sk int , c_customer_id varchar(16) , c_current_cdemo_sk int , c_current_hdemo_sk int , c_current_addr_sk int , c_first_shipto_date_sk int , c_first_sales_date_sk int , c_salutation varchar(10) , c_first_name varchar(20) , c_last_name varchar(30) , c_preferred_cust_flag varchar(1) , c_birth_day int , c_birth_month int , c_birth_year int , c_birth_country varchar(20) , c_login varchar(13) , c_email_address varchar(50) , c_last_review_date_sk int ) CLUSTER BY(c_customer_sk) ;",
    "create_customer_address": f"CREATE OR REPLACE TABLE {database}.{schema}.customer_address       ( ca_address_sk int , ca_address_id varchar(16) , ca_street_number varchar(10) , ca_street_name varchar(60) , ca_street_type varchar(15) , ca_suite_number varchar(10) , ca_city varchar(60) , ca_county varchar(30) , ca_state varchar(2) , ca_zip varchar(10) , ca_country varchar(20) , ca_gmt_offset decimal(5,2) , ca_location_type varchar(20) ) CLUSTER BY(ca_address_sk) ;",
    "create_customer_demographics": f"CREATE OR REPLACE TABLE {database}.{schema}.customer_demographics  ( cd_demo_sk int , cd_gender varchar(1) , cd_marital_status varchar(1) , cd_education_status varchar(20) , cd_purchase_estimate int , cd_credit_rating varchar(10) , cd_dep_count int , cd_dep_employed_count int , cd_dep_college_count int ) CLUSTER BY(cd_demo_sk) ;",
    "create_date_dim": f"CREATE OR REPLACE TABLE {database}.{schema}.date_dim               ( d_date_sk int , d_date_id varchar(16) , d_date date , d_month_seq int , d_week_seq int , d_quarter_seq int , d_year int , d_dow int , d_moy int , d_dom int , d_qoy int , d_fy_year int , d_fy_quarter_seq int , d_fy_week_seq int , d_day_name varchar(9) , d_quarter_name varchar(6) , d_holiday varchar(1) , d_weekend varchar(1) , d_following_holiday varchar(1) , d_first_dom int , d_last_dom int , d_same_day_ly int , d_same_day_lq int , d_current_day varchar(1) , d_current_week varchar(1) , d_current_month varchar(1) , d_current_quarter varchar(1) , d_current_year varchar(1) ) CLUSTER BY(d_date_sk) ;",
    "create_household_demographics": f"CREATE OR REPLACE TABLE {database}.{schema}.household_demographics ( hd_demo_sk int , hd_income_band_sk int , hd_buy_potential varchar(15) , hd_dep_count int , hd_vehicle_count int ) CLUSTER BY(hd_demo_sk) ;",
    "create_income_band": f"CREATE OR REPLACE TABLE {database}.{schema}.income_band            ( ib_income_band_sk int , ib_lower_bound int , ib_upper_bound int ) CLUSTER BY(ib_income_band_sk) ;",
    "create_inventory": f"CREATE OR REPLACE TABLE {database}.{schema}.inventory              ( inv_date_sk int , inv_item_sk int , inv_warehouse_sk int , inv_quantity_on_hand int ) CLUSTER BY(inv_date_sk) ;",
    "create_item": f"CREATE OR REPLACE TABLE {database}.{schema}.item                   ( i_item_sk int , i_item_id varchar(16) , i_rec_start_date date , i_rec_end_date date , i_item_desc varchar(200) , i_current_price decimal(7,2) , i_wholesale_cost decimal(7,2) , i_brand_id int , i_brand varchar(50) , i_class_id int , i_class varchar(50) , i_category_id int , i_category varchar(50) , i_manufact_id int , i_manufact varchar(50) , i_size varchar(20) , i_formulation varchar(20) , i_color varchar(20) , i_units varchar(10) , i_container varchar(10) , i_manager_id int , i_product_name varchar(50) ) CLUSTER BY(i_item_sk) ;",
    "create_promotion": f"CREATE OR REPLACE TABLE {database}.{schema}.promotion              ( p_promo_sk int , p_promo_id varchar(16) , p_start_date_sk int , p_end_date_sk int , p_item_sk int , p_cost decimal(15,2) , p_response_target int , p_promo_name varchar(50) , p_channel_dmail varchar(1) , p_channel_email varchar(1) , p_channel_catalog varchar(1) , p_channel_tv varchar(1) , p_channel_radio varchar(1) , p_channel_press varchar(1) , p_channel_event varchar(1) , p_channel_demo varchar(1) , p_channel_details varchar(100) , p_purpose varchar(15) , p_discount_active varchar(1) ) CLUSTER BY(p_promo_sk) ;",
    "create_reason": f"CREATE OR REPLACE TABLE {database}.{schema}.reason                 ( r_reason_sk int , r_reason_id varchar(16) , r_reason_desc varchar(100) ) CLUSTER BY(r_reason_sk) ;",
    "create_ship_mode": f"CREATE OR REPLACE TABLE {database}.{schema}.ship_mode              ( sm_ship_mode_sk int , sm_ship_mode_id varchar(16) , sm_type varchar(30) , sm_code varchar(10) , sm_carrier varchar(20) , sm_contract varchar(20) ) CLUSTER BY(sm_ship_mode_sk) ;",
    "create_store": f"CREATE OR REPLACE TABLE {database}.{schema}.store                  ( s_store_sk int , s_store_id varchar(16) , s_rec_start_date date , s_rec_end_date date , s_closed_date_sk int , s_store_name varchar(50) , s_number_employees int , s_floor_space int , s_hours varchar(20) , s_manager varchar(40) , s_market_id int , s_geography_class varchar(100) , s_market_desc varchar(100) , s_market_manager varchar(40) , s_division_id int , s_division_name varchar(50) , s_company_id int , s_company_name varchar(50) , s_street_number varchar(10) , s_street_name varchar(60) , s_street_type varchar(15) , s_suite_number varchar(10) , s_city varchar(60) , s_county varchar(30) , s_state varchar(2) , s_zip varchar(10) , s_country varchar(20) , s_gmt_offset decimal(5,2) , s_tax_precentage decimal(5,2) ) CLUSTER BY(s_store_sk) ;",
    "create_store_returns": f"CREATE OR REPLACE TABLE {database}.{schema}.store_returns          ( sr_returned_date_sk int , sr_return_time_sk int , sr_item_sk int , sr_customer_sk int , sr_cdemo_sk int , sr_hdemo_sk int , sr_addr_sk int , sr_store_sk int , sr_reason_sk int , sr_ticket_number bigint , sr_return_quantity int , sr_return_amt decimal(7,2) , sr_return_tax decimal(7,2) , sr_return_amt_inc_tax decimal(7,2) , sr_fee decimal(7,2) , sr_return_ship_cost decimal(7,2) , sr_refunded_cash decimal(7,2) , sr_reversed_charge decimal(7,2) , sr_store_credit decimal(7,2) , sr_net_loss decimal(7,2) ) CLUSTER BY(sr_returned_date_sk) ;",
    "create_store_sales": f"CREATE OR REPLACE TABLE {database}.{schema}.store_sales            ( ss_sold_date_sk int , ss_sold_time_sk int , ss_item_sk int , ss_customer_sk int , ss_cdemo_sk int , ss_hdemo_sk int , ss_addr_sk int , ss_store_sk int , ss_promo_sk int , ss_ticket_number bigint , ss_quantity int , ss_wholesale_cost decimal(7,2) , ss_list_price decimal(7,2) , ss_sales_price decimal(7,2) , ss_ext_discount_amt decimal(7,2) , ss_ext_sales_price decimal(7,2) , ss_ext_wholesale_cost decimal(7,2) , ss_ext_list_price decimal(7,2) , ss_ext_tax decimal(7,2) , ss_coupon_amt decimal(7,2) , ss_net_paid decimal(7,2) , ss_net_paid_inc_tax decimal(7,2) , ss_net_profit decimal(7,2) ) CLUSTER BY(ss_sold_date_sk) ;",
    "create_time_dim": f"CREATE OR REPLACE TABLE {database}.{schema}.time_dim               ( t_time_sk int , t_time_id varchar(16) , t_time int , t_hour int , t_minute int , t_second int , t_am_pm varchar(2) , t_shift varchar(20) , t_sub_shift varchar(20) , t_meal_time varchar(20) ) CLUSTER BY(t_time_sk) ;",
    "create_warehouse": f"CREATE OR REPLACE TABLE {database}.{schema}.warehouse              ( w_warehouse_sk int , w_warehouse_id varchar(16) , w_warehouse_name varchar(20) , w_warehouse_sq_ft int , w_street_number varchar(10) , w_street_name varchar(60) , w_street_type varchar(15) , w_suite_number varchar(10) , w_city varchar(60) , w_county varchar(30) , w_state varchar(2) , w_zip varchar(10) , w_country varchar(20) , w_gmt_offset decimal(5,2) ) CLUSTER BY(w_warehouse_sk) ;",
    "create_web_page": f"CREATE OR REPLACE TABLE {database}.{schema}.web_page               ( wp_web_page_sk int , wp_web_page_id varchar(16) , wp_rec_start_date date , wp_rec_end_date date , wp_creation_date_sk int , wp_access_date_sk int , wp_autogen_flag varchar(1) , wp_customer_sk int , wp_url varchar(100) , wp_type varchar(50) , wp_char_count int , wp_link_count int , wp_image_count int , wp_max_ad_count int ) CLUSTER BY(wp_web_page_sk) ;",
    "create_web_returns": f"CREATE OR REPLACE TABLE {database}.{schema}.web_returns            ( wr_returned_date_sk int , wr_returned_time_sk int , wr_item_sk int , wr_refunded_customer_sk int , wr_refunded_cdemo_sk int , wr_refunded_hdemo_sk int , wr_refunded_addr_sk int , wr_returning_customer_sk int , wr_returning_cdemo_sk int , wr_returning_hdemo_sk int , wr_returning_addr_sk int , wr_web_page_sk int , wr_reason_sk int , wr_order_number bigint , wr_return_quantity int , wr_return_amt decimal(7,2) , wr_return_tax decimal(7,2) , wr_return_amt_inc_tax decimal(7,2) , wr_fee decimal(7,2) , wr_return_ship_cost decimal(7,2) , wr_refunded_cash decimal(7,2) , wr_reversed_charge decimal(7,2) , wr_account_credit decimal(7,2) , wr_net_loss decimal(7,2) ) CLUSTER BY(wr_returned_date_sk) ;",
    "create_web_sales": f"CREATE OR REPLACE TABLE {database}.{schema}.web_sales              ( ws_sold_date_sk int , ws_sold_time_sk int , ws_ship_date_sk int , ws_item_sk int , ws_bill_customer_sk int , ws_bill_cdemo_sk int , ws_bill_hdemo_sk int , ws_bill_addr_sk int , ws_ship_customer_sk int , ws_ship_cdemo_sk int , ws_ship_hdemo_sk int , ws_ship_addr_sk int , ws_web_page_sk int , ws_web_site_sk int , ws_ship_mode_sk int , ws_warehouse_sk int , ws_promo_sk int , ws_order_number bigint , ws_quantity int , ws_wholesale_cost decimal(7,2) , ws_list_price decimal(7,2) , ws_sales_price decimal(7,2) , ws_ext_discount_amt decimal(7,2) , ws_ext_sales_price decimal(7,2) , ws_ext_wholesale_cost decimal(7,2) , ws_ext_list_price decimal(7,2) , ws_ext_tax decimal(7,2) , ws_coupon_amt decimal(7,2) , ws_ext_ship_cost decimal(7,2) , ws_net_paid decimal(7,2) , ws_net_paid_inc_tax decimal(7,2) , ws_net_paid_inc_ship decimal(7,2) , ws_net_paid_inc_ship_tax decimal(7,2) , ws_net_profit decimal(7,2) ) CLUSTER BY(ws_sold_date_sk) ;",
    "create_web_site": f"CREATE OR REPLACE TABLE {database}.{schema}.web_site               ( web_site_sk int , web_site_id varchar(16) , web_rec_start_date date , web_rec_end_date date , web_name varchar(50) , web_open_date_sk int , web_close_date_sk int , web_class varchar(50) , web_manager varchar(40) , web_mkt_id int , web_mkt_class varchar(50) , web_mkt_desc varchar(100) , web_market_manager varchar(40) , web_company_id int , web_company_name varchar(50) , web_street_number varchar(10) , web_street_name varchar(60) , web_street_type varchar(15) , web_suite_number varchar(10) , web_city varchar(60) , web_county varchar(30) , web_state varchar(2) , web_zip varchar(10) , web_country varchar(20) , web_gmt_offset decimal(5,2) , web_tax_percentage decimal(5,2) ) CLUSTER BY(web_site_sk) ;",
    "copy_call_center": f"COPY INTO {database}.{schema}.call_center            FROM @{stage}/{scale}/call_center/            PATTERN = '.*dat' FORCE = TRUE;",
    "copy_catalog_page": f"COPY INTO {database}.{schema}.catalog_page           FROM @{stage}/{scale}/catalog_page/           PATTERN = '.*dat' FORCE = TRUE;",
    "copy_catalog_returns": f"COPY INTO {database}.{schema}.catalog_returns        FROM @{stage}/{scale}/catalog_returns/        PATTERN = '.*dat' FORCE = TRUE;",
    "copy_catalog_sales": f"COPY INTO {database}.{schema}.catalog_sales          FROM @{stage}/{scale}/catalog_sales/          PATTERN = '.*dat' FORCE = TRUE;",
    "copy_customer": f"COPY INTO {database}.{schema}.customer               FROM @{stage}/{scale}/customer/               PATTERN = '.*dat' FORCE = TRUE;",
    "copy_customer_address": f"COPY INTO {database}.{schema}.customer_address       FROM @{stage}/{scale}/customer_address/       PATTERN = '.*dat' FORCE = TRUE;",
    "copy_customer_demographics": f"COPY INTO {database}.{schema}.customer_demographics  FROM @{stage}/{scale}/customer_demographics/  PATTERN = '.*dat' FORCE = TRUE;",
    "copy_date_dim": f"COPY INTO {database}.{schema}.date_dim               FROM @{stage}/{scale}/date_dim/               PATTERN = '.*dat' FORCE = TRUE;",
    "copy_household_demographics": f"COPY INTO {database}.{schema}.household_demographics FROM @{stage}/{scale}/household_demographics/ PATTERN = '.*dat' FORCE = TRUE;",
    "copy_income_band": f"COPY INTO {database}.{schema}.income_band            FROM @{stage}/{scale}/income_band/            PATTERN = '.*dat' FORCE = TRUE;",
    "copy_inventory": f"COPY INTO {database}.{schema}.inventory              FROM @{stage}/{scale}/inventory/              PATTERN = '.*dat' FORCE = TRUE;",
    "copy_item": f"COPY INTO {database}.{schema}.item                   FROM @{stage}/{scale}/item/                   PATTERN = '.*dat' FORCE = TRUE;",
    "copy_promotion": f"COPY INTO {database}.{schema}.promotion              FROM @{stage}/{scale}/promotion/              PATTERN = '.*dat' FORCE = TRUE;",
    "copy_reason": f"COPY INTO {database}.{schema}.reason                 FROM @{stage}/{scale}/reason/                 PATTERN = '.*dat' FORCE = TRUE;",
    "copy_ship_mode": f"COPY INTO {database}.{schema}.ship_mode              FROM @{stage}/{scale}/ship_mode/              PATTERN = '.*dat' FORCE = TRUE;",
    "copy_store": f"COPY INTO {database}.{schema}.store                  FROM @{stage}/{scale}/store/                  PATTERN = '.*dat' FORCE = TRUE;",
    "copy_store_returns": f"COPY INTO {database}.{schema}.store_returns          FROM @{stage}/{scale}/store_returns/          PATTERN = '.*dat' FORCE = TRUE;",
    "copy_store_sales": f"COPY INTO {database}.{schema}.store_sales            FROM @{stage}/{scale}/store_sales/            PATTERN = '.*dat' FORCE = TRUE;",
    "copy_time_dim": f"COPY INTO {database}.{schema}.time_dim               FROM @{stage}/{scale}/time_dim/               PATTERN = '.*dat' FORCE = TRUE;",
    "copy_warehouse": f"COPY INTO {database}.{schema}.warehouse              FROM @{stage}/{scale}/warehouse/              PATTERN = '.*dat' FORCE = TRUE;",
    "copy_web_page": f"COPY INTO {database}.{schema}.web_page               FROM @{stage}/{scale}/web_page/               PATTERN = '.*dat' FORCE = TRUE;",
    "copy_web_returns": f"COPY INTO {database}.{schema}.web_returns            FROM @{stage}/{scale}/web_returns/            PATTERN = '.*dat' FORCE = TRUE;",
    "copy_web_sales": f"COPY INTO {database}.{schema}.web_sales              FROM @{stage}/{scale}/web_sales/              PATTERN = '.*dat' FORCE = TRUE;",
    "copy_web_site": f"COPY INTO {database}.{schema}.web_site               FROM @{stage}/{scale}/web_site/               PATTERN = '.*dat' FORCE = TRUE;",
}

# Snowflake clustering columns - same as Databricks zorder strategy
# (unified across all strategies for consistent comparison)
sf_clustering_columns = {
    "call_center": "cc_call_center_sk, cc_name, cc_county",
    "catalog_page": "cp_catalog_page_sk",
    "catalog_returns": "cr_item_sk, cr_order_number, cr_returned_date_sk",
    "catalog_sales": "cs_sold_date_sk, cs_item_sk, cs_bill_customer_sk",
    "customer": "c_customer_sk, c_current_addr_sk, c_current_cdemo_sk",
    "customer_address": "ca_address_sk, ca_state, ca_county",
    "customer_demographics": "cd_demo_sk, cd_marital_status, cd_education_status",
    "date_dim": "d_date_sk, d_year, d_moy",
    "household_demographics": "hd_dep_count, hd_vehicle_count, hd_demo_sk",
    "income_band": "ib_income_band_sk",
    "inventory": "inv_item_sk, inv_date_sk, inv_warehouse_sk",
    "item": "i_item_sk, i_category, i_current_price",
    "promotion": "p_promo_sk",
    "reason": "r_reason_sk",
    "ship_mode": "sm_ship_mode_sk, sm_carrier",
    "store": "s_store_sk, s_state, s_zip",
    "store_returns": "sr_item_sk, sr_ticket_number, sr_customer_sk",
    "store_sales": "ss_sold_date_sk, ss_item_sk, ss_store_sk",
    "time_dim": "t_hour, t_minute, t_time_sk",
    "warehouse": "w_warehouse_sk",
    "web_page": "wp_web_page_sk, wp_char_count",
    "web_returns": "wr_item_sk, wr_order_number, wr_returned_date_sk",
    "web_sales": "ws_sold_date_sk, ws_item_sk, ws_bill_customer_sk",
    "web_site": "web_site_sk",
}

# Update CLUSTER BY clauses to use multi-column clustering
import re as _re
for key, stmt in ddl_statements.items():
    if key.startswith("create_"):
        tbl_name = key.removeprefix("create_")
        if tbl_name in sf_clustering_columns:
            new_cluster = f"CLUSTER BY({sf_clustering_columns[tbl_name]})"
            ddl_statements[key] = _re.sub(r'CLUSTER BY\([^)]+\)', new_cluster, stmt)

# Convert to Iceberg tables if requested
sf_strategy = params.get("sf_strategy", "native_gen1").strip().lower()
sf_strategy_base = "iceberg" if "iceberg" in sf_strategy else "native"
sf_warehouse_gen = 2 if "gen2" in sf_strategy else 1
if sf_strategy_base == "iceberg":
    # External volume + base location come from the database-level default set by
    # terraform: `CREATE DATABASE ... EXTERNAL_VOLUME = 'tpcds_iceberg_volume'`.
    import re as _re2

    for key, stmt in ddl_statements.items():
        if key.startswith("create_"):
            # Remove CLUSTER BY clause (not supported on Iceberg)
            stmt = _re2.sub(r'\s*CLUSTER BY\([^)]+\)', '', stmt)
            # Change CREATE OR REPLACE TABLE to CREATE OR REPLACE ICEBERG TABLE
            stmt = stmt.replace('CREATE OR REPLACE TABLE', 'CREATE OR REPLACE ICEBERG TABLE')
            # Add CATALOG = 'SNOWFLAKE' before the column list.
            # EXTERNAL_VOLUME / BASE_LOCATION are inherited from the DB default.
            stmt = _re2.sub(
                r'(ICEBERG TABLE\s+\S+\.\S+\.\S+\s*)(\()',
                r"\1 CATALOG = 'SNOWFLAKE' \2",
                stmt
            )

            ddl_statements[key] = stmt
    # Snowflake Iceberg doesn't support VARCHAR(N) - only VARCHAR or VARCHAR(134217728)
    import re as _re3
    for key, stmt in ddl_statements.items():
        if key.startswith("create_"):
            ddl_statements[key] = _re3.sub(r'varchar\(\d+\)', 'varchar', stmt)
    print("Converted to Iceberg tables (external_volume inherited from DB default)")


# --- Merge CREATE + COPY into a single CTAS (matches Databricks CTAS semantics) ---
# After this block, ddl_statements holds only create_<t> entries, each of which
# both CREATEs the table with explicit schema+NOT NULL and populates it from
# the external stage in one statement. Gives us 1:1 q_id parity with Databricks
# for cross-platform join by q_id.

import re as _re_ctas


def _split_top_level(s, delim=","):
    depth, cur, parts = 0, [], []
    for ch in s:
        if ch == "(": depth += 1; cur.append(ch)
        elif ch == ")": depth -= 1; cur.append(ch)
        elif ch == delim and depth == 0:
            parts.append("".join(cur)); cur = []
        else: cur.append(ch)
    if cur: parts.append("".join(cur))
    return [x.strip() for x in parts if x.strip()]


def _cast_list_from_col_defs(col_block):
    # Returns "$1::int, $2::varchar(16), ..." preserving per-column types
    casts = []
    for i, col in enumerate(_split_top_level(col_block), start=1):
        tokens = col.split(None, 2)
        col_type = tokens[1]
        casts.append(f"${i}::{col_type}")
    return ", ".join(casts)


_ctas_statements = {}
for _key in [k for k in ddl_statements.keys() if k.startswith("create_")]:
    _tbl = _key.removeprefix("create_")
    _create = ddl_statements[_key]
    _m = _re_ctas.search(r"\((?P<cols>[^()]*(?:\([^()]*\)[^()]*)*)\)\s*(?:CLUSTER|AS|;|$)", _create)
    # Fallback: match the LAST (...) before a terminal clause
    if not _m:
        _m = _re_ctas.search(r"\(([^()]*(?:\([^()]*\)[^()]*)*)\)", _create)
        _cols = _m.group(1) if _m else ""
    else:
        _cols = _m.group("cols")
    _select = _cast_list_from_col_defs(_cols)

    _copy = ddl_statements.get(f"copy_{_tbl}", "")
    _path = _re_ctas.search(r"FROM\s+(@\S+)", _copy)
    _stage_path = _path.group(1) if _path else f"@{stage}/{scale}/{_tbl}/"
    _pat = _re_ctas.search(r"PATTERN\s*=\s*'([^']+)'", _copy)
    _pattern_clause = f" (PATTERN => '{_pat.group(1)}')" if _pat else ""

    _ctas = _create.rstrip().rstrip(";").rstrip() + f" AS SELECT {_select} FROM {_stage_path}{_pattern_clause};"
    _ctas_statements[_key] = _ctas

ddl_statements = dict(_ctas_statements)
print(f"Merged into {len(_ctas_statements)} CTAS statements (q_id vocabulary: create_<table>)")

# Apply table name prefix if set
table_prefix = params.get("table_prefix", "")
if table_prefix:
    import re as _re
    def _prefix_sf_ddl(sql, prefix):
        # Prefix table name in CREATE OR REPLACE TABLE {db}.{schema}.{table}
        sql = _re.sub(
            r'(CREATE OR REPLACE (?:ICEBERG )?TABLE\s+\S+\.\S+\.)(\w+)',
            lambda m: m.group(1) + prefix + m.group(2),
            sql
        )
        # Prefix table name in COPY INTO {db}.{schema}.{table} but NOT in FROM @stage path
        sql = _re.sub(
            r'(COPY INTO\s+\S+\.\S+\.)(\w+)',
            lambda m: m.group(1) + prefix + m.group(2),
            sql
        )
        return sql
    ddl_statements = {name: _prefix_sf_ddl(stmt, table_prefix) for name, stmt in ddl_statements.items()}
    print(f"Table names prefixed with '{table_prefix}'")

In [ ]:
import os
table_prefix = params.get("table_prefix", "").rstrip("_")
volume_base = f"/Volumes/{params['dbx_catalog']}/{params['dbx_schema']}/{params['dbx_volume']}"
conf_dir = f"{volume_base}/test_files/{table_prefix}" if table_prefix else f"{volume_base}/test_files"
os.makedirs(conf_dir, exist_ok=True)
for f in os.listdir(conf_dir):
    if f.endswith("-snowflake.tsv"):
        os.remove(os.path.join(conf_dir, f))
print(f"Config dir: {conf_dir}")

In [ ]:
persist_to_tsv(ddl_statements, f'{conf_dir}/tpcds-queries-ingestion-snowflake.tsv')

In [ ]:
# Snowflake auto-clusters and auto-collects column stats — no OPTIMIZE/ANALYZE
# phase needed; the JMeter INGESTION_TEMPLATE has only Phase 1 (CTAS) anyway.

In [ ]:
import csv as _csv, shutil as _shutil

table_prefix = params.get("table_prefix", "")
if table_prefix:
    rows = []
    with open('./resources/tpcds-queries-snowflake.tsv', 'r') as f:
        reader = _csv.reader(f, delimiter='\t', quoting=_csv.QUOTE_NONE)
        for row in reader:
            rows.append(row)
    for i in range(1, len(rows)):
        rows[i][1] = prefix_table_names(rows[i][1], table_prefix)
    with open(f'{conf_dir}/tpcds-queries-perf-snowflake.tsv', 'w', newline='') as f:
        writer = _csv.writer(f, delimiter='\t', quoting=_csv.QUOTE_NONE, escapechar=None, quotechar=None)
        writer.writerows(rows)
    print(f"Perf queries prefixed with '{table_prefix}'")
else:
    _shutil.copy('./resources/tpcds-queries-snowflake.tsv', f'{conf_dir}/tpcds-queries-perf-snowflake.tsv')
    print("Perf queries copied (no prefix)")